# GRPO on GSM8K: Colab quick run

Runs a short GRPO training (LoRA, Qwen2.5-0.5B-Instruct) on a free T4 and compares it with the zero-shot model. Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

In [ ]:
%cd /content
import os
if not os.path.exists('/content/grpo-gsm8k'):  # safe to re-run
    !git clone https://github.com/aymenblk04/grpo-gsm8k.git
%cd /content/grpo-gsm8k
!git pull -q
!pip install -q -r requirements.txt && pip install -q -e .
# Colab ships torchao 0.10; peft refuses to import with torchao < 0.16 and we don't use it
!pip uninstall -q -y torchao

`run()` streams a command's output and **stops the notebook if it fails**, so an error shows up where it happens.

In [ ]:
import subprocess

def run(cmd):
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'command failed (exit {p.returncode}): {cmd}')

T4 has no bf16 support, so we switch to fp16 and shorten everything (v2 rewards, see README). Training takes ~45 min.

In [ ]:
COLAB = ['grpo.bf16=false', 'grpo.fp16=true', 'grpo.max_steps=100',
         'grpo.num_generations=4', 'grpo.per_device_train_batch_size=4',
         'grpo.max_completion_length=300', 'data.train_samples=500']
run('python scripts/train_grpo.py --config configs/base.yaml output_dir=outputs/v2 ' + ' '.join(COLAB))

In [ ]:
run('python scripts/evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --num-samples 200 --max-new-tokens 400 --out results/zero_shot')
run('python scripts/evaluate.py --model Qwen/Qwen2.5-0.5B-Instruct --adapter outputs/v2 --num-samples 200 --max-new-tokens 400 --out results/v2')

In [ ]:
run('python scripts/plot.py --runs v2')
from IPython.display import Image, Markdown, display
display(Markdown(open('figures/results.md').read()))
display(Image('figures/training_curves.png'))

In [ ]:
import json
for line in list(open('results/v2/samples.jsonl'))[:3]:
    r = json.loads(line)
    print('gold:', r['gold'], '| pred:', r['prediction'], '| correct:', r['correct'])
    print(r['completion'][:500], '\n' + '-' * 80)

Download the results (metrics, samples, figures, training log).

In [ ]:
run('zip -qr results.zip results figures outputs/v2/trainer_state.json')
from google.colab import files
files.download('results.zip')